## Milestone 3
### Feature Engineering
### Yiscah Mark


In [146]:
from google.colab import files

uploaded = files.upload()

Saving cleaned_car_data.csv to cleaned_car_data (9).csv


Displaying the data. Throughout I keep displaying the first five rows to see the changes.

In [147]:
# Import necessary libraries
import pandas as pd
import numpy as np

# Load the dataset
df = pd.read_csv("CAR DETAILS FROM CAR DEKHO.csv")
display(df.head())


,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner
0,Maruti 800 AC,2007,60000,70000,Petrol,Individual,Manual,First Owner
1,Maruti Wagon R LXI Minor,2007,135000,50000,Petrol,Individual,Manual,First Owner
2,Hyundai Verna 1.6 SX,2012,600000,100000,Diesel,Individual,Manual,First Owner
3,Datsun RediGO T Option,2017,250000,46000,Petrol,Individual,Manual,First Owner
4,Honda Amaze VX i-DTEC,2014,450000,141000,Diesel,Individual,Manual,Second Owner


Checking the shape, the datatype in each column, the column names...

In [148]:
# Check the initial shape and info
print(df.shape)
df.head()
df.info()
df.describe()
df.isna()

(4340, 8)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4340 entries, 0 to 4339
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   name           4340 non-null   object
 1   year           4340 non-null   int64 
 2   selling_price  4340 non-null   int64 
 3   km_driven      4340 non-null   int64 
 4   fuel           4340 non-null   object
 5   seller_type    4340 non-null   object
 6   transmission   4340 non-null   object
 7   owner          4340 non-null   object
dtypes: int64(3), object(5)
memory usage: 271.4+ KB


,name,year,selling_price,km_driven,fuel,seller_type,transmission,owner
0,False,False,False,False,False,False,False,False
1,False,False,False,False,False,False,False,False
2,False,False,False,False,False,False,False,False
3,False,False,False,False,False,False,False,False
4,False,False,False,False,False,False,False,False
...,...,...,...,...,...,...,...,...
4335,False,False,False,False,False,False,False,False
4336,False,False,False,False,False,False,False,False
4337,False,False,False,False,False,False,False,False
4338,False,False,False,False,False,False,False,False


When I counted the fuel types in the fuel column some of the petrol fuel were spelled with a capital P and some with a lowercase p, therefore I standardized the text to lowercase. I also reprinted the revised totals and I see that the electric and lpg have such low counts that it just acts as noise.

In [149]:
# 1. Standardize the text to lowercase BEFORE encoding
df['fuel'] = df['fuel'].str.lower()

# 2. Re-run the encoding
# We will also drop 'electric' manually or let it be handled by drop_first
df = pd.get_dummies(df, columns=['fuel'], drop_first=True, dtype=int)

# 3. Check the counts again
fuel_cols = [col for col in df.columns if col.startswith('fuel')]
print(df[fuel_cols].sum())

fuel_diesel      2153
fuel_electric       1
fuel_lpg           23
fuel_petrol      2123
dtype: int64


I removed the electric and LPG columns, and then rechecked the new totals.


In [150]:
# Filter the dataframe to keep only Diesel and Petrol based on dummy variables
# This removes the 1 Electric and 23 LPG rows
# Keep rows where fuel_electric is 0 AND fuel_lpg is 0.
df = df[(df['fuel_electric'] == 0) & (df['fuel_lpg'] == 0)]

# The original 'fuel' column no longer exists, so standardizing casing is not applicable.
# Verify the clean counts by summing the dummy variables
print("Remaining fuel type counts:")
print(f"Diesel: {df['fuel_diesel'].sum()}")
print(f"Petrol: {df['fuel_petrol'].sum()}")
print(f"Electric: {df['fuel_electric'].sum()}")
print(f"LPG: {df['fuel_lpg'].sum()}")

Remaining fuel type counts:
Diesel: 2153
Petrol: 2123
Electric: 0
LPG: 0


In [151]:
display(df.head())

,name,year,selling_price,km_driven,seller_type,transmission,owner,fuel_diesel,fuel_electric,fuel_lpg,fuel_petrol
0,Maruti 800 AC,2007,60000,70000,Individual,Manual,First Owner,0,0,0,1
1,Maruti Wagon R LXI Minor,2007,135000,50000,Individual,Manual,First Owner,0,0,0,1
2,Hyundai Verna 1.6 SX,2012,600000,100000,Individual,Manual,First Owner,1,0,0,0
3,Datsun RediGO T Option,2017,250000,46000,Individual,Manual,First Owner,0,0,0,1
4,Honda Amaze VX i-DTEC,2014,450000,141000,Individual,Manual,Second Owner,1,0,0,0


There are too many unique names of cars. This is too noizy for the model. The company is enough to get a picture and predict. I rewrote a new name column wich just pulls out the first word of each name, which is the company.

I then printed all teh unique names.

In [152]:
# Extract the first word from the 'name' column and reassign it to replace the original
df['name'] = df['name'].str.split().str[0]

# Verify the change
print(df['name'].unique())
display(df.head())

['Maruti' 'Hyundai' 'Datsun' 'Honda' 'Tata' 'Chevrolet' 'Toyota' 'Jaguar'
 'Mercedes-Benz' 'Audi' 'Skoda' 'Jeep' 'BMW' 'Mahindra' 'Ford' 'Nissan'
 'Renault' 'Fiat' 'Volkswagen' 'Volvo' 'Mitsubishi' 'Land' 'Daewoo' 'MG'
 'Force' 'Isuzu' 'OpelCorsa' 'Ambassador' 'Kia']


,name,year,selling_price,km_driven,seller_type,transmission,owner,fuel_diesel,fuel_electric,fuel_lpg,fuel_petrol
0,Maruti,2007,60000,70000,Individual,Manual,First Owner,0,0,0,1
1,Maruti,2007,135000,50000,Individual,Manual,First Owner,0,0,0,1
2,Hyundai,2012,600000,100000,Individual,Manual,First Owner,1,0,0,0
3,Datsun,2017,250000,46000,Individual,Manual,First Owner,0,0,0,1
4,Honda,2014,450000,141000,Individual,Manual,Second Owner,1,0,0,0


Here I printed the unique car company names along with the counts.

In [153]:
print("Number of unique car brands:", df['name'].nunique())
print("car brands:\n", df['name'].value_counts().head(29))

Number of unique car brands: 29
car brands:
 name
Maruti           1266
Hyundai           813
Mahindra          365
Tata              361
Honda             252
Ford              238
Toyota            205
Chevrolet         187
Renault           146
Volkswagen        107
Skoda              68
Nissan             64
Audi               60
BMW                39
Fiat               37
Datsun             37
Mercedes-Benz      35
Mitsubishi          6
Jaguar              6
Land                5
Volvo               4
Ambassador          4
Jeep                3
OpelCorsa           2
MG                  2
Force               1
Daewoo              1
Isuzu               1
Kia                 1
Name: count, dtype: int64


Some conpanies appear very rearly. In a way they act as noise and mess up the model. On the other hand we lose information. I decided to put the threshold at 10. Any car company that appears less that 10 times is removed.

In [154]:
# Get the counts of each car brand
brand_counts = df['name'].value_counts()

# Identify brands with less than 10 occurrences
rare_brands = brand_counts[brand_counts < 10].index

# Filter the DataFrame to keep only brands with 10 or more occurrences
df = df[~df['name'].isin(rare_brands)]

print(f"Number of car brands before filtering: {len(brand_counts)}")
print(f"Number of car brands after filtering: {df['name'].nunique()}")
print("Remaining car brands (top 20):\n", df['name'].value_counts().head(20))

display(df.head())

Number of car brands before filtering: 29
Number of car brands after filtering: 17
Remaining car brands (top 20):
 name
Maruti           1266
Hyundai           813
Mahindra          365
Tata              361
Honda             252
Ford              238
Toyota            205
Chevrolet         187
Renault           146
Volkswagen        107
Skoda              68
Nissan             64
Audi               60
BMW                39
Datsun             37
Fiat               37
Mercedes-Benz      35
Name: count, dtype: int64


,name,year,selling_price,km_driven,seller_type,transmission,owner,fuel_diesel,fuel_electric,fuel_lpg,fuel_petrol
0,Maruti,2007,60000,70000,Individual,Manual,First Owner,0,0,0,1
1,Maruti,2007,135000,50000,Individual,Manual,First Owner,0,0,0,1
2,Hyundai,2012,600000,100000,Individual,Manual,First Owner,1,0,0,0
3,Datsun,2017,250000,46000,Individual,Manual,First Owner,0,0,0,1
4,Honda,2014,450000,141000,Individual,Manual,Second Owner,1,0,0,0


I have two numerical columns (year and Kilometers driven). The year only goes to about two thousand and the kilometers goes over 100,000. Since these ranges are so different it can mess up the model. In the next cell I will normalize the figures by making each a number between 0 and 1 so the difference in range doesn't skew everything.I used the min-max scaling in which 0 is the lowest number in ther range and 1 is the highest. I didn't use standard scaling, which scales relative the the standard deviations because i wanted it to be in the same range as the encodded columns.

I am obviously not normalizing my selling price column because it is the target variable.

In [155]:
from sklearn.preprocessing import MinMaxScaler

# Initialize the scaler
scaler = MinMaxScaler()

# Select the columns to normalize
cols_to_fix = ['km_driven', 'year']

# Apply the transformation
df[cols_to_fix] = scaler.fit_transform(df[cols_to_fix])

display(df.head())

,name,year,selling_price,km_driven,seller_type,transmission,owner,fuel_diesel,fuel_electric,fuel_lpg,fuel_petrol
0,Maruti,0.535714,60000,0.086783,Individual,Manual,First Owner,0,0,0,1
1,Maruti,0.535714,135000,0.061988,Individual,Manual,First Owner,0,0,0,1
2,Hyundai,0.714286,600000,0.123976,Individual,Manual,First Owner,1,0,0,0
3,Datsun,0.892857,250000,0.057028,Individual,Manual,First Owner,0,0,0,1
4,Honda,0.785714,450000,0.174807,Individual,Manual,Second Owner,1,0,0,0


I am encoding these categoricl tada columns so i can feed it into a regression model.

In [156]:
categorical_cols_to_encode = ['seller_type', 'transmission', 'owner']

# Apply one-hot encoding
df = pd.get_dummies(df, columns=categorical_cols_to_encode, drop_first=True, dtype=int)

display(df.head())
print(f"Shape after encoding: {df.shape}")

,name,year,selling_price,km_driven,fuel_diesel,fuel_electric,fuel_lpg,fuel_petrol,seller_type_Individual,seller_type_Trustmark Dealer,transmission_Manual,owner_Fourth & Above Owner,owner_Second Owner,owner_Test Drive Car,owner_Third Owner
0,Maruti,0.535714,60000,0.086783,0,0,0,1,1,0,1,0,0,0,0
1,Maruti,0.535714,135000,0.061988,0,0,0,1,1,0,1,0,0,0,0
2,Hyundai,0.714286,600000,0.123976,1,0,0,0,1,0,1,0,0,0,0
3,Datsun,0.892857,250000,0.057028,0,0,0,1,1,0,1,0,0,0,0
4,Honda,0.785714,450000,0.174807,1,0,0,0,1,0,1,0,1,0,0


Shape after encoding: (4280, 15)


I am examining the rest of the encoded categories to see if I should drop anything.

In [157]:
print("Current columns in DataFrame:")
print(df.columns.tolist())

print("\n--- Examining Categorical Feature Distributions (after one-hot encoding) ---")

# Assuming these columns were one-hot encoded with drop_first=True
# We'll sum the dummy variables to get a sense of their distribution.

# Seller Type
seller_type_dummies = [col for col in df.columns if col.startswith('seller_type_')]
if seller_type_dummies:
    print("\nSeller Type Distribution (from dummy variables):")
    display(df[seller_type_dummies].sum().rename(lambda x: x.replace('seller_type_', '')))
else:
    print("\nNo seller_type dummy columns found. Please verify preprocessing steps.")

# Transmission Type
transmission_dummies = [col for col in df.columns if col.startswith('transmission_')]
if transmission_dummies:
    print("\nTransmission Type Distribution (from dummy variables):")
    display(df[transmission_dummies].sum().rename(lambda x: x.replace('transmission_', '')))
else:
    print("\nNo transmission_type dummy columns found. Please verify preprocessing steps.")

# Owner Type
owner_dummies = [col for col in df.columns if col.startswith('owner_')]
if owner_dummies:
    print("\nOwner Type Distribution (from dummy variables):")
    display(df[owner_dummies].sum().rename(lambda x: x.replace('owner_', '')))
else:
    print("\nNo owner_type dummy columns found. Please verify preprocessing steps.")

print("\nCar Brand (Name) Distribution (after rare brand removal):")
display(df['name'].value_counts())

Current columns in DataFrame:
['name', 'year', 'selling_price', 'km_driven', 'fuel_diesel', 'fuel_electric', 'fuel_lpg', 'fuel_petrol', 'seller_type_Individual', 'seller_type_Trustmark Dealer', 'transmission_Manual', 'owner_Fourth & Above Owner', 'owner_Second Owner', 'owner_Test Drive Car', 'owner_Third Owner']

--- Examining Categorical Feature Distributions (after one-hot encoding) ---

Seller Type Distribution (from dummy variables):


,0
Individual,3196
Trustmark Dealer,102



Transmission Type Distribution (from dummy variables):


,0
Manual,3851



Owner Type Distribution (from dummy variables):


,0
Fourth & Above Owner,78
Second Owner,1084
Test Drive Car,17
Third Owner,300



Car Brand (Name) Distribution (after rare brand removal):


,count
name,
Maruti,1266
Hyundai,813
Mahindra,365
Tata,361
Honda,252
Ford,238
Toyota,205
Chevrolet,187
Renault,146


Everything looks within range. I will leave it.

Since most of the cars are priced regularly and we have a few ouliers that are totaly out of range, I will perform log transformation on my price column. This "shows" the model that predicting $5,000 off on a $25,000 car is a lot worse than oon a $100,000 car. The log transformation normalizes the selling price in this way.

In [166]:
# Apply the log transformation
# df['price_log'] = np.log(df['selling_price'])

# Compare the skewness (Optional check)
# print(f"Original Skew: {df['selling_price'].skew()}") # This line will also cause an error if selling_price is dropped
print(f"Log-Transformed Skew: {df['price_log'].skew()}")

display(df.head())

Log-Transformed Skew: 0.03797551145232248


,year,km_driven,fuel_diesel,fuel_electric,fuel_lpg,fuel_petrol,seller_type_Individual,seller_type_Trustmark Dealer,transmission_Manual,owner_Fourth & Above Owner,...,name_Hyundai,name_Mahindra,name_Maruti,name_Mercedes-Benz,name_Nissan,name_Renault,name_Skoda,name_Tata,name_Toyota,name_Volkswagen
0,0.535714,0.086783,0,0,0,1,1,0,1,0,...,0,0,1,0,0,0,0,0,0,0
1,0.535714,0.061988,0,0,0,1,1,0,1,0,...,0,0,1,0,0,0,0,0,0,0
2,0.714286,0.123976,1,0,0,0,1,0,1,0,...,1,0,0,0,0,0,0,0,0,0
3,0.892857,0.057028,0,0,0,1,1,0,1,0,...,0,0,0,0,0,0,0,0,0,0
4,0.785714,0.174807,1,0,0,0,1,0,1,0,...,0,0,0,0,0,0,0,0,0,0


I didn't want to one-hot encode the names column because I didn't like how so many names that mean something just became a series of ones and zeros. The only way to feed it to a regression model is if it is numerical so here it is.

In [164]:
# Apply one-hot encoding to the 'name' column
df = pd.get_dummies(df, columns=['name'], drop_first=True, dtype=int)

display(df.head())
print(f"Shape after encoding 'name': {df.shape}")

,year,km_driven,fuel_diesel,fuel_electric,fuel_lpg,fuel_petrol,seller_type_Individual,seller_type_Trustmark Dealer,transmission_Manual,owner_Fourth & Above Owner,...,name_Hyundai,name_Mahindra,name_Maruti,name_Mercedes-Benz,name_Nissan,name_Renault,name_Skoda,name_Tata,name_Toyota,name_Volkswagen
0,0.535714,0.086783,0,0,0,1,1,0,1,0,...,0,0,1,0,0,0,0,0,0,0
1,0.535714,0.061988,0,0,0,1,1,0,1,0,...,0,0,1,0,0,0,0,0,0,0
2,0.714286,0.123976,1,0,0,0,1,0,1,0,...,1,0,0,0,0,0,0,0,0,0
3,0.892857,0.057028,0,0,0,1,1,0,1,0,...,0,0,0,0,0,0,0,0,0,0
4,0.785714,0.174807,1,0,0,0,1,0,1,0,...,0,0,0,0,0,0,0,0,0,0


Shape after encoding 'name': (4280, 30)


I am dropping the original price column so we don't get confused.

In [165]:
cols_to_drop = ['selling_price']

# Drop them permanently
df.drop(columns=cols_to_drop, inplace=True, errors='ignore')

display(df.head())

,year,km_driven,fuel_diesel,fuel_electric,fuel_lpg,fuel_petrol,seller_type_Individual,seller_type_Trustmark Dealer,transmission_Manual,owner_Fourth & Above Owner,...,name_Hyundai,name_Mahindra,name_Maruti,name_Mercedes-Benz,name_Nissan,name_Renault,name_Skoda,name_Tata,name_Toyota,name_Volkswagen
0,0.535714,0.086783,0,0,0,1,1,0,1,0,...,0,0,1,0,0,0,0,0,0,0
1,0.535714,0.061988,0,0,0,1,1,0,1,0,...,0,0,1,0,0,0,0,0,0,0
2,0.714286,0.123976,1,0,0,0,1,0,1,0,...,1,0,0,0,0,0,0,0,0,0
3,0.892857,0.057028,0,0,0,1,1,0,1,0,...,0,0,0,0,0,0,0,0,0,0
4,0.785714,0.174807,1,0,0,0,1,0,1,0,...,0,0,0,0,0,0,0,0,0,0


I was going to make the kilometers driven into bins, like split into categories. I didn't since I normalized it and it works well as a continuous number.

This is the last step to prepare for regression, by splitting the features and the target variable into x and y.

In [167]:
X = df.drop('price_log', axis=1)
y = df['price_log']

print("Shape of X (features):", X.shape)
print("Shape of y (target):", y.shape)

display(X.head())
display(y.head())

Shape of X (features): (4280, 29)
Shape of y (target): (4280,)


,year,km_driven,fuel_diesel,fuel_electric,fuel_lpg,fuel_petrol,seller_type_Individual,seller_type_Trustmark Dealer,transmission_Manual,owner_Fourth & Above Owner,...,name_Hyundai,name_Mahindra,name_Maruti,name_Mercedes-Benz,name_Nissan,name_Renault,name_Skoda,name_Tata,name_Toyota,name_Volkswagen
0,0.535714,0.086783,0,0,0,1,1,0,1,0,...,0,0,1,0,0,0,0,0,0,0
1,0.535714,0.061988,0,0,0,1,1,0,1,0,...,0,0,1,0,0,0,0,0,0,0
2,0.714286,0.123976,1,0,0,0,1,0,1,0,...,1,0,0,0,0,0,0,0,0,0
3,0.892857,0.057028,0,0,0,1,1,0,1,0,...,0,0,0,0,0,0,0,0,0,0
4,0.785714,0.174807,1,0,0,0,1,0,1,0,...,0,0,0,0,0,0,0,0,0,0


,price_log
0,11.002100
1,11.813030
2,13.304685
3,12.429216
4,13.017003
